# S0b - pandas survival kit

## Learning objectives

By the end of this notebook you can:

1. load a CSV into a DataFrame and describe its shape and columns
2. select one column, several columns, and a subset of rows
3. filter rows by a condition and count how many matched
4. compute summary statistics, and read `describe()` output
5. group rows by a category and compare group averages
6. find missing values and say how many there are

## Prerequisites

- `S0a - Python for data`, in particular lists, `for`, `if` and reading a traceback.
- The data pack is built (`python build/prepare_data.py --auto`).

**Time:** about 1 hour

## Engineering context

The AEOLIS component factory runs a bank of milling machines. Every production run is logged with the machine's air and process temperature, rotational speed, torque, accumulated tool wear, and whether the machine failed.

That log is 10,000 rows. This is the real dataset you will model in Sessions 2 and 3 to decide when to send a technician - so the questions you ask of it here are the same questions you will ask then.

## Why this matters

In `S0a` a day of readings was a list of six numbers. Real logs have 10,000 rows and 14 columns, and a `for` loop over them is both slow to write and slow to run.

pandas replaces the loop with a question. `df[df['Torque [Nm]'] > 60]` is "show me the high-torque runs" - one line, no loop, no counter. Nearly everything you do in the six sessions is a variation on the eleven operations below.

**Learn these eleven and you can follow every practical in the course.**

In [ ]:
# Run me first. Sets the course up on Colab; does nothing if you are already local.
COURSE_REPO = ""
DRIVE_FOLDER = "AEOLIS_course"


def _setup():
    import importlib, os, subprocess, sys
    from pathlib import Path

    try:
        importlib.import_module("aeolis")
        return "already available"
    except ModuleNotFoundError:
        pass

    for candidate in [Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}"), Path("/content/course")]:
        if (candidate / "aeolis").is_dir():
            os.chdir(candidate)
            sys.path.insert(0, str(candidate))
            return f"found at {candidate}"

    if COURSE_REPO:
        target = Path("/content/course")
        subprocess.run(["git", "clone", "--depth", "1", COURSE_REPO, str(target)], check=True)
        os.chdir(target)
        sys.path.insert(0, str(target))
        return "cloned from the course repository"

    raise SystemExit(
        "The course files are not here yet. Do ONE of these, then re-run this cell:\n\n"
        "  1. Ask your instructor for the repository link and run, in a new cell:\n"
        "       !git clone <link> /content/course\n\n"
        f"  2. Put the course folder in your Google Drive as '{DRIVE_FOLDER}', then run:\n"
        "       from google.colab import drive; drive.mount('/content/drive')\n\n"
        "  3. Run this notebook on your own computer, inside the course folder."
    )


print("course:", _setup())

import aeolis
from pathlib import Path

needed = ["data/ai4i.csv", "data/consumption.parquet"]
missing = [f for f in needed if not Path(f).exists()]
if missing:
    print("data pack: MISSING", missing)
    print("           the instructor runs: python build/prepare_data.py --auto")
else:
    print("data pack: ok")
print("ready")

## Code Companion

Every exercise below can be solved by adapting one of these. Copy, change the names, run. That is not cheating - it is how data code actually gets written.

```python
# load a table from disk
df = pd.read_csv('data/ai4i.csv')

# look at the first few rows
df.head()

# how big is it? (rows, columns)
df.shape

# what are the columns called?
df.columns

# one column (a Series)
df['Torque [Nm]']

# several columns (a DataFrame)
df[['Torque [Nm]', 'Tool wear [min]']]

# summary statistics for every column
df.describe()

# one statistic
df['Torque [Nm]'].mean()

# keep only the rows matching a condition
hot = df[df['Torque [Nm]'] > 60]

# count each value of a category
df['Type'].value_counts()

# average per group
df.groupby('Type')['Torque [Nm]'].mean()

# missing values per column
df.isna().sum()

# sort by a column, biggest first
df.sort_values('Tool wear [min]', ascending=False)
```

## 1. Load it, then look at it

The first thing you do with any dataset - every time, without exception - is find out how big it is and what is in it. Session 1 makes a discipline of this; here it is just three lines.

In [ ]:
import pandas as pd

df = pd.read_csv('data/ai4i.csv')
print('rows and columns:', df.shape)
df.head()

`df` is a **DataFrame**: a table with named columns, like a spreadsheet you can ask questions of. `.head()` shows the first five rows.

Note the column names carefully: `'Air temperature [K]'` includes the unit **and** the spaces. You have to type them exactly. That is annoying and it is also real - industrial data arrives named by whoever configured the logger.

In [ ]:
list(df.columns)

In [ ]:
df.dtypes

`dtypes` says what kind of value each column holds: `float64` for measurements, `int64` for whole numbers and the 0/1 failure flags, `object` for text.

**Why it matters:** a numeric column that arrives as `object` almost always means dirty data - a stray `'N/A'` or a comma decimal separator somewhere in 10,000 rows. Checking `dtypes` is how you find it in one second instead of one hour. You will do exactly this in Session 1.

**Your turn.**

In [ ]:
# TODO (1): how many rows does the log have?
n_rows = ...

# TODO (2): how many columns?
n_cols = ...

**Hint 1.** `df.shape` gives both, as a pair. Pull them out the way you indexed a list in S0a.

**Hint 2.** `df.shape[0]` is the number of rows; `df.shape[1]` the number of columns.

In [ ]:
from aeolis.checks import done
done(n_rows, n_cols)
assert n_rows == 10000, f'expected 10000 rows, got {n_rows}'
assert n_cols == 14, f'expected 14 columns, got {n_cols}'
print('correct')

## 2. Selecting columns

One set of square brackets gives you a single column - a **Series**. Two sets give you a smaller table - a **DataFrame**. The difference matters later, so meet it now.

In [ ]:
torque = df['Torque [Nm]']         # one column -> Series
print(type(torque).__name__)
print(torque.head(3))

In [ ]:
pair = df[['Torque [Nm]', 'Tool wear [min]']]   # two columns -> DataFrame
print(type(pair).__name__)
pair.head(3)

A Series behaves like a very long, very fast list of numbers, and it comes with the statistics attached:

In [ ]:
print('mean   :', torque.mean())
print('min    :', torque.min())
print('max    :', torque.max())
print('std dev:', torque.std())

Compare that with `S0a`, where you wrote a loop and a counter to get an average. Same answer, one word, and it works on 10,000 rows as fast as on six.

In [ ]:
# TODO: the average tool wear, in minutes
mean_wear = ...

In [ ]:
from aeolis.checks import done, close
done(mean_wear)
assert close(mean_wear, 107.95, tol=0.5), f'expected about 108, got {mean_wear}'
print('correct')

## 3. `describe()`: the thirty-second overview

One call, every numeric column, eight statistics. This is the single most useful line in pandas and the first thing you will run in Session 1.

In [ ]:
df.describe()

How to read it, row by row:

- **count** - how many non-missing values. If this differs between columns, you have missing data.
- **mean** and **std** - the centre and the spread.
- **min** / **max** - the extremes. Look here first: a temperature of -999 or a speed of 0 is usually a broken sensor, not a broken machine.
- **25% / 50% / 75%** - the quartiles. The 50% row is the **median**.

**Mean versus median** is worth ten seconds now, because Session 1 leans on it. If they are close, the values are fairly symmetric. If the mean is much higher than the median, a few large values are dragging it up - and an average that a handful of extreme readings controls is an average that will mislead you.

In [ ]:
speed = df['Rotational speed [rpm]']
print(f'mean   {speed.mean():.1f} rpm')
print(f'median {speed.median():.1f} rpm')
print(f'max    {speed.max():.1f} rpm   <- pulling the mean up')

## 4. Filtering rows: asking a question

This is the operation that replaces the loop. Inside the brackets you put a condition; pandas keeps the rows where it is true.

In [ ]:
failures = df[df['Machine failure'] == 1]
print('runs that ended in failure:', len(failures))
print(f'that is {len(failures) / len(df) * 100:.2f}% of all runs')
failures.head(3)

**3.39%.** Remember that number - it is small, and in Session 2 that smallness becomes the most important fact about this dataset. A model that simply always predicts "no failure" would be right 96.6% of the time and completely useless. Building your intuition for *why* is most of Session 2.

Conditions combine with `&` (and) and `|` (or). Each condition needs its own brackets:

In [ ]:
hot_and_worn = df[(df['Process temperature [K]'] > 310) & (df['Tool wear [min]'] > 200)]
print('hot AND worn:', len(hot_and_worn))

**Your turn.**

In [ ]:
# TODO (1): the runs where torque exceeded 60 Nm
high_torque = ...

# TODO (2): how many were there?
n_high_torque = ...

**Hint 1.** The Code Companion pattern is `hot = df[df['Torque [Nm]'] > 60]`. For the count, `len()`.

In [ ]:
from aeolis.checks import done
done(high_torque, n_high_torque)
assert n_high_torque == 236, f'expected 236, got {n_high_torque}'
print('correct')

## 5. Grouping: comparing categories

The `Type` column records the product quality grade: L (low), M (medium), H (high). First, how many of each?

In [ ]:
df['Type'].value_counts()

Now the useful question: **does the failure rate differ by grade?** `groupby` splits the rows into groups and computes one number per group.

In [ ]:
by_type = df.groupby('Type')['Machine failure'].mean()
print(by_type)
print()
print('as percentages:')
print((by_type * 100).round(2))

Taking the `mean` of a 0/1 column gives you the **proportion of 1s** - the failure rate. That trick appears constantly.

**Read the result like an engineer, not like a spreadsheet.** The rates differ between grades. Does that mean the grade *causes* failures? Not necessarily - the grades may be run at different speeds, on different jobs, for different durations. Session 1 has a whole segment on exactly this trap. For now: notice the difference, resist the explanation.

In [ ]:
# TODO: the average tool wear for each product grade
wear_by_type = ...

**Hint 1.** Same shape as the failure-rate line above, with a different column and statistic.

In [ ]:
from aeolis.checks import done
done(wear_by_type)
assert len(wear_by_type) == 3, f'expected 3 groups, got {len(wear_by_type)}'
assert set(wear_by_type.index) == {'H', 'L', 'M'}, 'groups should be H, L and M'
print('correct')
print(wear_by_type.round(1))

## 6. Missing values

`isna()` marks every missing cell, and `.sum()` counts the marks per column. Run it on every dataset before you do anything else.

In [ ]:
df.isna().sum()

All zeros. This dataset is synthetic and tidy, which is exactly why Sessions 2 and 3 use it - the lesson there is modelling, not cleaning.

Session 1 uses a different dataset: real turbine sensor readings, with missing runs, stuck sensors and duplicated timestamps. **Real data does not look like this.** Enjoy it while it lasts.

In [ ]:
# what missing data actually looks like: the Session 4 consumption series
consumption = pd.read_parquet('data/consumption.parquet')
print('rows:', len(consumption))
print(consumption.isna().sum())
print(f"\nmissing: {consumption['consumption_kw'].isna().mean() * 100:.2f}% of hours")

1.22% of hours have no reading. Not a disaster, and not nothing - and in Session 4 you will have to decide what to do about it, because a forecasting model cannot be handed a gap.

## 7. Sorting: finding the extremes

Sorting is how you look at the worst cases, which is usually where the interesting engineering is.

In [ ]:
df.sort_values('Tool wear [min]', ascending=False)[
    ['Type', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']
].head(5)

Read that line right to left: sort by tool wear, biggest first; keep four columns; show the top five. Chaining operations like this is the normal pandas style, and it is why one line does what fifteen lines of loop would.

## Knowledge check

Five tasks, no hints, one line each. Same shape as the Session 1 entry check.

In [ ]:
# 1. The highest process temperature in the log
max_process_temp = ...

# 2. The number of runs with tool wear above 240 minutes
n_worn_out = ...

# 3. The median torque
median_torque = ...

# 4. The average rotational speed for each product grade (a Series of 3 values)
speed_by_type = ...

# 5. The number of runs flagged with a tool-wear failure (column 'TWF')
n_twf = ...

In [ ]:
from aeolis.checks import done, close
done(max_process_temp, n_worn_out, median_torque, speed_by_type, n_twf)
score = 0
score += close(max_process_temp, 313.8, tol=0.2)
score += (n_worn_out == 11)
score += close(median_torque, 40.1, tol=0.3)
score += (len(speed_by_type) == 3)
score += (n_twf == 46)
print(f'score: {score}/5')
print('ready for Session 1' if score >= 4 else 'review the sections you missed, then retry')

## Reflection

1. In section 4 you found that 3.39% of runs end in failure. Suppose you built a model that always answers "no failure". What percentage of the time would it be right - and would you deploy it?
2. Which single pandas operation do you expect to use most in Session 1?

In [ ]:
reflection = """
1.
2.
"""

## Summary

- `read_csv` / `read_parquet` load a table; `shape`, `columns`, `dtypes`, `head()` tell you what you have.
- `df['col']` is one column (a Series); `df[['a','b']]` is a smaller table.
- `describe()` is the thirty-second overview. Read the min and max rows first.
- `df[df['col'] > x]` filters rows - this replaces the loop.
- `groupby('cat')['col'].mean()` compares groups; the mean of a 0/1 column is a rate.
- `isna().sum()` counts missing values. Run it before anything else.

## Common mistakes

- **`and` instead of `&`.** pandas needs `&` and `|`, and each condition in its own brackets.
- **Column name not exact.** `'Torque'` is not `'Torque [Nm]'`. Copy from `list(df.columns)`.
- **`df['a','b']`.** Two columns need two sets of brackets: `df[['a','b']]`.
- **Filtering and forgetting to assign.** `df[df['x'] > 1]` returns a new table; it does not change `df`.
- **Reading a groupby difference as a cause.** Groups differ for many reasons. Notice it; do not explain it yet.

## Further exploration (optional)

- `df.corr()` shows how columns move together - Session 1 uses it, with a warning attached.
- `df.to_csv('out.csv', index=False)` saves your work. You will need it when you submit notebooks.
- pandas has hundreds of methods. You need the eleven above; look the rest up when a real question demands one.

---

**Next:** `S0c - plots and arrays`, where the table becomes a picture - because you will read far more plots in this course than you will write.